In [1]:
# ============================================================
# INDIAN LEGAL MAMBA LANGUAGE MODEL
# Improved Training Version
#
# Goal:
#   Improve validation perplexity of the Mamba model.
#
# Dataset:
#   indian-legal-slm-tokenized-2048
#
# Model:
#   Mamba
#   d_model = 512
#   6 layers
#   d_state = 16
#   d_conv = 4
#
# IMPORTANT:
#   This trains Mamba FROM SCRATCH.
# ============================================================


# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

!pip install -q -U mambapy datasets tokenizers


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import math
import json
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk
from tokenizers import Tokenizer

from mambapy.mamba import Mamba, MambaConfig


# ============================================================
# 3. ENVIRONMENT
# ============================================================

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using:", DEVICE)


# ============================================================
# 4. REPRODUCIBILITY
# ============================================================

SEED = 61

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True


# ============================================================
# 5. PATHS
# ============================================================

BASE_PATH = (
    "/kaggle/input/datasets/"
    "belovedorange/indian-legal-slm-tokenized-2048"
)

DATASET_PATH = os.path.join(
    BASE_PATH,
    "indian_legal_2048"
)

TOKENIZER_PATH = os.path.join(
    BASE_PATH,
    "indian_legal_tokenizer",
    "tokenizer.json"
)

print("\nDataset path:")
print(DATASET_PATH)

print("\nTokenizer path:")
print(TOKENIZER_PATH)


# ============================================================
# 6. LOAD DATASET
# ============================================================

dataset = load_from_disk(
    DATASET_PATH
)

print("\nDataset:")
print(dataset)


# ============================================================
# 7. LOAD TOKENIZER
# ============================================================

tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)

VOCAB_SIZE = tokenizer.get_vocab_size()

PAD_ID = tokenizer.token_to_id(
    "<pad>"
)

BOS_ID = tokenizer.token_to_id(
    "<bos>"
)

EOS_ID = tokenizer.token_to_id(
    "<eos>"
)

print("\nTokenizer:")
print("Vocabulary size:", VOCAB_SIZE)
print("PAD:", PAD_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)


# ============================================================
# 8. CHECK DATA
# ============================================================

example = dataset["train"][0]["input_ids"]

print("\nExample sequence length:")
print(len(example))

print("\nFirst 20 token IDs:")
print(example[:20])

print("\nDecoded example:")
print(
    tokenizer.decode(
        example[:500]
    )
)


# ============================================================
# 9. MODEL HYPERPARAMETERS
# ============================================================

CONTEXT_LENGTH = 2048

D_MODEL = 512
NUM_LAYERS = 6
D_STATE = 16
D_CONV = 4

print("\n")
print("=" * 70)
print("MODEL CONFIGURATION")
print("=" * 70)

print(
    "Context length:",
    CONTEXT_LENGTH
)

print(
    "d_model:",
    D_MODEL
)

print(
    "Layers:",
    NUM_LAYERS
)

print(
    "d_state:",
    D_STATE
)

print(
    "d_conv:",
    D_CONV
)


# ============================================================
# 10. TRAINING HYPERPARAMETERS
# ============================================================

BATCH_SIZE = 1

GRAD_ACCUMULATION_STEPS = 16

EPOCHS = 3

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.03

MAX_GRAD_NORM = 1.0

LOG_EVERY = 100

CHECKPOINT_EVERY = 5000


print("\n")
print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRAD_ACCUMULATION_STEPS
)

print(
    "Effective batch size:",
    BATCH_SIZE * GRAD_ACCUMULATION_STEPS
)

print(
    "Epochs:",
    EPOCHS
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "Warmup ratio:",
    WARMUP_RATIO
)


# ============================================================
# 11. DATASET WRAPPER
# ============================================================

class LegalDataset(Dataset):

    def __init__(
        self,
        hf_dataset
    ):

        self.dataset = hf_dataset

    def __len__(self):

        return len(
            self.dataset
        )

    def __getitem__(
        self,
        idx
    ):

        return self.dataset[
            idx
        ]["input_ids"]


# ============================================================
# 12. CREATE TRAIN / VALIDATION DATASETS
# ============================================================

# FULL DATASET
train_dataset = LegalDataset(
    dataset["train"]
)

val_dataset = LegalDataset(
    dataset["validation"]
)

# train_dataset = LegalDataset(
#     dataset["train"].select(range(10))
# )

# val_dataset = LegalDataset(
#     dataset["validation"].select(range(10))
# )



# ------------------------------------------------------------
# FOR A 10-ROW SANITY TEST ONLY:
#
# train_dataset = LegalDataset(
#     dataset["train"].select(range(10))
# )
#
# val_dataset = LegalDataset(
#     dataset["validation"].select(range(10))
# )
# ------------------------------------------------------------


print("\nTrain sequences:")
print(len(train_dataset))

print(
    "Validation sequences:"
)
print(
    len(val_dataset)
)


# ============================================================
# 13. COLLATE FUNCTION
# ============================================================

def collate_fn(batch):

    max_length = max(
        len(sequence)
        for sequence in batch
    )

    input_ids = torch.full(
        (
            len(batch),
            max_length
        ),
        PAD_ID,
        dtype=torch.long
    )

    for i, sequence in enumerate(
        batch
    ):

        input_ids[
            i,
            :len(sequence)
        ] = torch.tensor(
            sequence,
            dtype=torch.long
        )

    return input_ids


# ============================================================
# 14. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


print("\nTrain batches:")
print(len(train_loader))

print("\nValidation batches:")
print(len(val_loader))


# ============================================================
# 15. MAMBA LANGUAGE MODEL
# ============================================================

class MambaLanguageModel(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        d_state,
        d_conv
    ):

        super().__init__()

        self.d_model = d_model

        # Token embedding
        self.embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_ID
        )

        # Mamba configuration
        mamba_config = MambaConfig(
            d_model=d_model,
            n_layers=num_layers,
            d_state=d_state,
            d_conv=d_conv
        )

        self.mamba = Mamba(
            mamba_config
        )

        # Final normalization
        self.norm = nn.LayerNorm(
            d_model
        )

        # Language-model head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Tie embedding and output weights
        self.lm_head.weight = (
            self.embedding.weight
        )


    def forward(
        self,
        input_ids
    ):

        x = self.embedding(
            input_ids
        )

        x = self.mamba(
            x
        )

        x = self.norm(
            x
        )

        logits = self.lm_head(
            x
        )

        return logits


# ============================================================
# 16. CREATE MODEL
# ============================================================

model = MambaLanguageModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_layers=NUM_LAYERS,
    d_state=D_STATE,
    d_conv=D_CONV
)

model = model.to(
    DEVICE
)


# ============================================================
# 17. PARAMETER COUNT
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n")
print("=" * 70)
print("MODEL PARAMETERS")
print("=" * 70)

print(
    f"Total parameters:     {total_params:,}"
)

print(
    f"Trainable parameters: {trainable_params:,}"
)


# ============================================================
# 18. SANITY CHECK
# ============================================================

batch = next(
    iter(train_loader)
)

batch = batch.to(
    DEVICE,
    non_blocking=True
)

print("\nInput shape:")
print(batch.shape)


with torch.no_grad():

    logits = model(
        batch
    )


print("\nOutput shape:")
print(logits.shape)

assert (
    logits.shape[0]
    ==
    batch.shape[0]
)

assert (
    logits.shape[1]
    ==
    batch.shape[1]
)

assert (
    logits.shape[2]
    ==
    VOCAB_SIZE
)

print(
    "\nSanity check PASSED."
)


# ============================================================
# 19. LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)


# ============================================================
# 20. OPTIMIZER
# ============================================================

try:

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.95),
        eps=1e-8,
        fused=(
            DEVICE.type == "cuda"
        )
    )

    print(
        "\nUsing fused AdamW."
    )

except Exception:

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        betas=(0.9, 0.95),
        eps=1e-8
    )

    print(
        "\nUsing standard AdamW."
    )


# ============================================================
# 21. TRAINING STEP CALCULATION
# ============================================================

steps_per_epoch = math.ceil(
    len(train_loader)
    /
    GRAD_ACCUMULATION_STEPS
)

TOTAL_STEPS = (
    steps_per_epoch
    *
    EPOCHS
)

WARMUP_STEPS = max(
    1,
    int(
        TOTAL_STEPS
        *
        WARMUP_RATIO
    )
)


print("\n")
print("=" * 70)
print("TRAINING STEPS")
print("=" * 70)

print(
    "Steps per epoch:",
    steps_per_epoch
)

print(
    "Total optimizer steps:",
    TOTAL_STEPS
)

print(
    "Warmup steps:",
    WARMUP_STEPS
)


# ============================================================
# 22. COSINE LEARNING-RATE SCHEDULER
# ============================================================

def get_lr_multiplier(
    current_step
):

    # Warmup
    if current_step < WARMUP_STEPS:

        return (
            float(
                current_step + 1
            )
            /
            float(
                WARMUP_STEPS
            )
        )

    # Cosine decay
    progress = (
        current_step
        -
        WARMUP_STEPS
    ) / max(
        1,
        TOTAL_STEPS
        -
        WARMUP_STEPS
    )

    progress = min(
        1.0,
        max(
            0.0,
            progress
        )
    )

    return (
        0.5
        *
        (
            1.0
            +
            math.cos(
                math.pi
                *
                progress
            )
        )
    )


scheduler = (
    torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=get_lr_multiplier
    )
)


# ============================================================
# 23. AMP SCALER
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        DEVICE.type == "cuda"
    )
)


# ============================================================
# 24. CHECKPOINT DIRECTORY
# ============================================================

MODEL_DIR = (
    "/kaggle/working/"
    "indian_legal_mamba_improved"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "best_model.pt"
)

FINAL_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "mamba_model.pt"
)

CHECKPOINT_PATH = os.path.join(
    MODEL_DIR,
    "latest_checkpoint.pt"
)

CONFIG_PATH = os.path.join(
    MODEL_DIR,
    "config.json"
)


# ============================================================
# 25. TRAINING FUNCTION
# ============================================================

def train_one_epoch(
    epoch
):

    model.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    epoch_loss_sum = 0.0

    epoch_batches = 0

    recent_losses = []

    optimizer_steps_this_epoch = 0

    start_time = time.time()


    for step, batch in enumerate(
        train_loader
    ):

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Causal LM setup
        # ----------------------------------------------------

        inputs = batch[:, :-1]

        labels = batch[:, 1:]


        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(
                DEVICE.type == "cuda"
            )
        ):

            # Mamba is causal by design.
            #
            # No causal attention mask
            # is required.

            logits = model(
                inputs
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                labels.reshape(
                    -1
                )
            )


        loss_value = (
            loss.detach().item()
        )


        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        epoch_loss_sum += (
            loss_value
        )

        epoch_batches += 1

        recent_losses.append(
            loss_value
        )

        if len(
            recent_losses
        ) > 1000:

            recent_losses.pop(0)


        # ----------------------------------------------------
        # Gradient accumulation
        # ----------------------------------------------------

        loss_for_backward = (
            loss
            /
            GRAD_ACCUMULATION_STEPS
        )

        scaler.scale(
            loss_for_backward
        ).backward()


        should_step = (
            (step + 1)
            %
            GRAD_ACCUMULATION_STEPS
            ==
            0
        )

        is_last_batch = (
            (step + 1)
            ==
            len(train_loader)
        )


        # ----------------------------------------------------
        # Optimizer step
        # ----------------------------------------------------

        if (
            should_step
            or
            is_last_batch
        ):

            scaler.unscale_(
                optimizer
            )

            grad_norm = (
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM
                )
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            optimizer_steps_this_epoch += 1


            current_lr = (
                optimizer.param_groups[
                    0
                ]["lr"]
            )


            global_optimizer_step = (
                epoch
                *
                steps_per_epoch
                +
                optimizer_steps_this_epoch
            )


            # ------------------------------------------------
            # Logging
            # ------------------------------------------------

            if (
                optimizer_steps_this_epoch
                %
                LOG_EVERY
                ==
                0
            ):

                recent_mean = (
                    sum(
                        recent_losses
                    )
                    /
                    len(
                        recent_losses
                    )
                )

                elapsed = (
                    time.time()
                    -
                    start_time
                )

                print(
                    f"Epoch {epoch + 1}/{EPOCHS} | "
                    f"Opt Step "
                    f"{global_optimizer_step:,}/"
                    f"{TOTAL_STEPS:,} | "
                    f"Batch "
                    f"{step + 1:,}/"
                    f"{len(train_loader):,} | "
                    f"Loss: "
                    f"{loss_value:.4f} | "
                    f"Recent Loss: "
                    f"{recent_mean:.4f} | "
                    f"LR: "
                    f"{current_lr:.3e} | "
                    f"Grad Norm: "
                    f"{float(grad_norm):.3f} | "
                    f"Time: "
                    f"{elapsed / 60:.1f} min"
                )


            # ------------------------------------------------
            # Periodic checkpoint
            # ------------------------------------------------

            if (
                global_optimizer_step
                %
                CHECKPOINT_EVERY
                ==
                0
            ):

                torch.save(
                    {
                        "epoch":
                            epoch,

                        "batch_step":
                            step,

                        "optimizer_step":
                            global_optimizer_step,

                        "model_state_dict":
                            model.state_dict(),

                        "optimizer_state_dict":
                            optimizer.state_dict(),

                        "scheduler_state_dict":
                            scheduler.state_dict(),

                        "scaler_state_dict":
                            scaler.state_dict(),

                        "train_loss":
                            loss_value
                    },
                    CHECKPOINT_PATH
                )

                print(
                    f"\nCheckpoint saved at "
                    f"optimizer step "
                    f"{global_optimizer_step:,}\n"
                )


    # --------------------------------------------------------
    # Epoch statistics
    # --------------------------------------------------------

    epoch_loss = (
        epoch_loss_sum
        /
        epoch_batches
    )

    recent_loss = (
        sum(recent_losses)
        /
        len(recent_losses)
    )

    return (
        epoch_loss,
        recent_loss
    )


# ============================================================
# 26. VALIDATION
# ============================================================

@torch.no_grad()
def evaluate():

    model.eval()

    total_loss = 0.0

    total_batches = 0


    for batch in val_loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        inputs = batch[:, :-1]

        labels = batch[:, 1:]


        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(
                DEVICE.type == "cuda"
            )
        ):

            logits = model(
                inputs
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                labels.reshape(
                    -1
                )
            )


        total_loss += (
            loss.item()
        )

        total_batches += 1


    return (
        total_loss
        /
        total_batches
    )


# ============================================================
# 27. PERPLEXITY
# ============================================================

def calculate_perplexity(
    loss
):

    return math.exp(
        min(
            loss,
            20
        )
    )


# ============================================================
# 28. TRAINING HISTORY
# ============================================================

train_losses = []

recent_train_losses = []

val_losses = []

train_ppls = []

val_ppls = []

learning_rates = []

best_val_loss = float(
    "inf"
)

best_epoch = -1


# ============================================================
# 29. START TRAINING
# ============================================================

print("\n")
print("=" * 70)
print("STARTING IMPROVED MAMBA TRAINING")
print("=" * 70)

training_start = time.time()


for epoch in range(
    EPOCHS
):

    print("\n")
    print("=" * 70)

    print(
        f"Epoch {epoch + 1}/{EPOCHS}"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    train_loss, recent_loss = (
        train_one_epoch(
            epoch
        )
    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    print(
        "\nRunning validation..."
    )

    val_loss = evaluate()


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    train_ppl = (
        calculate_perplexity(
            train_loss
        )
    )

    recent_train_ppl = (
        calculate_perplexity(
            recent_loss
        )
    )

    val_ppl = (
        calculate_perplexity(
            val_loss
        )
    )

    current_lr = (
        optimizer.param_groups[
            0
        ]["lr"]
    )


    # --------------------------------------------------------
    # Store history
    # --------------------------------------------------------

    train_losses.append(
        train_loss
    )

    recent_train_losses.append(
        recent_loss
    )

    val_losses.append(
        val_loss
    )

    train_ppls.append(
        train_ppl
    )

    val_ppls.append(
        val_ppl
    )

    learning_rates.append(
        current_lr
    )


    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    print("\n")
    print("-" * 70)

    print(
        f"Epoch {epoch + 1} RESULTS"
    )

    print("-" * 70)

    print(
        f"Train loss:          "
        f"{train_loss:.4f}"
    )

    print(
        f"Recent train loss:   "
        f"{recent_loss:.4f}"
    )

    print(
        f"Validation loss:     "
        f"{val_loss:.4f}"
    )

    print(
        f"Train perplexity:    "
        f"{train_ppl:.2f}"
    )

    print(
        f"Recent train PPL:    "
        f"{recent_train_ppl:.2f}"
    )

    print(
        f"Validation PPL:      "
        f"{val_ppl:.2f}"
    )

    print(
        f"Learning rate:       "
        f"{current_lr:.3e}"
    )


    # --------------------------------------------------------
    # BEST MODEL = BEST VALIDATION LOSS
    # --------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = (
            val_loss
        )

        best_epoch = (
            epoch
        )

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print(
            "\n*** NEW BEST MAMBA MODEL SAVED ***"
        )

        print(
            f"Best validation loss: "
            f"{best_val_loss:.4f}"
        )

        print(
            f"Best validation PPL: "
            f"{calculate_perplexity(best_val_loss):.2f}"
        )

    else:

        print(
            "\nValidation loss did not improve."
        )

    print(
        "-" * 70
    )


# ============================================================
# 30. TRAINING COMPLETE
# ============================================================

training_time = (
    time.time()
    -
    training_start
)

print("\n")
print("=" * 70)
print("MAMBA TRAINING COMPLETE")
print("=" * 70)

print(
    f"Total training time: "
    f"{training_time / 3600:.2f} hours"
)

print(
    f"Best epoch: "
    f"{best_epoch + 1}"
)

print(
    f"Best validation loss: "
    f"{best_val_loss:.4f}"
)

print(
    f"Best validation PPL: "
    f"{calculate_perplexity(best_val_loss):.2f}"
)


# ============================================================
# 31. LOAD BEST MODEL
# ============================================================

print(
    "\nLoading best validation model..."
)

best_state_dict = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

model.load_state_dict(
    best_state_dict
)

model.eval()

print(
    "Best model loaded."
)


# ============================================================
# 32. FINAL VALIDATION
# ============================================================

print(
    "\nRunning final validation..."
)

final_val_loss = evaluate()

final_val_ppl = (
    calculate_perplexity(
        final_val_loss
    )
)


print("\n")
print("=" * 70)
print("FINAL BEST MAMBA MODEL METRICS")
print("=" * 70)

print(
    f"Validation loss:       "
    f"{final_val_loss:.4f}"
)

print(
    f"Validation perplexity: "
    f"{final_val_ppl:.2f}"
)

print("=" * 70)


# ============================================================
# 33. SAVE FINAL MODEL
# ============================================================

torch.save(
    model.state_dict(),
    FINAL_MODEL_PATH
)

print(
    "\nFinal model saved to:"
)

print(
    FINAL_MODEL_PATH
)


# ============================================================
# 34. SAVE CONFIGURATION
# ============================================================

config = {

    "architecture":
        "Mamba Language Model",

    "vocab_size":
        VOCAB_SIZE,

    "context_length":
        CONTEXT_LENGTH,

    "d_model":
        D_MODEL,

    "num_layers":
        NUM_LAYERS,

    "d_state":
        D_STATE,

    "d_conv":
        D_CONV,

    "weight_tying":
        True,

    "batch_size":
        BATCH_SIZE,

    "gradient_accumulation_steps":
        GRAD_ACCUMULATION_STEPS,

    "effective_batch_size":
        (
            BATCH_SIZE
            *
            GRAD_ACCUMULATION_STEPS
        ),

    "epochs":
        EPOCHS,

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "warmup_ratio":
        WARMUP_RATIO,

    "max_grad_norm":
        MAX_GRAD_NORM,

    "seed":
        SEED,

    "best_epoch":
        best_epoch + 1,

    "best_validation_loss":
        final_val_loss,

    "best_validation_perplexity":
        final_val_ppl
}


with open(
    CONFIG_PATH,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )


print(
    "\nConfiguration saved to:"
)

print(
    CONFIG_PATH
)


# ============================================================
# 35. TRAINING LOSS CURVE
# ============================================================

epochs_axis = np.arange(
    1,
    EPOCHS + 1
)


plt.figure(
    figsize=(10, 6)
)

plt.plot(
    epochs_axis,
    train_losses,
    marker="o",
    label="Train Loss"
)

plt.plot(
    epochs_axis,
    recent_train_losses,
    marker="o",
    label="Recent Train Loss"
)

plt.plot(
    epochs_axis,
    val_losses,
    marker="o",
    label="Validation Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "Mamba Training and Validation Loss"
)

plt.xticks(
    epochs_axis
)

plt.grid(
    True,
    alpha=0.3
)

plt.legend()

plt.show()


# ============================================================
# 36. PERPLEXITY CURVE
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    epochs_axis,
    train_ppls,
    marker="o",
    label="Train Perplexity"
)

plt.plot(
    epochs_axis,
    val_ppls,
    marker="o",
    label="Validation Perplexity"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Perplexity"
)

plt.title(
    "Mamba Perplexity"
)

plt.xticks(
    epochs_axis
)

plt.grid(
    True,
    alpha=0.3
)

plt.legend()

plt.show()


# ============================================================
# 37. LEARNING RATE CURVE
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    epochs_axis,
    learning_rates,
    marker="o"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Learning Rate"
)

plt.title(
    "Mamba Learning Rate"
)

plt.xticks(
    epochs_axis
)

plt.grid(
    True,
    alpha=0.3
)

plt.show()


# ============================================================
# 38. GENERATION FUNCTION
# ============================================================

@torch.no_grad()
def generate(
    prompt,
    max_new_tokens=150,
    temperature=0.8,
    top_k=50,
    top_p=0.95
):

    model.eval()


    # --------------------------------------------------------
    # Encode prompt
    # --------------------------------------------------------

    prompt_ids = tokenizer.encode(
        prompt
    ).ids


    if len(prompt_ids) == 0:

        raise ValueError(
            "Prompt produced no tokens."
        )


    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE
    )


    # --------------------------------------------------------
    # Generate
    # --------------------------------------------------------

    for _ in range(
        max_new_tokens
    ):

        input_ids = input_ids[
            :,
            -CONTEXT_LENGTH:
        ]


        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(
                DEVICE.type == "cuda"
            )
        ):

            logits = model(
                input_ids
            )


        # Last-token logits
        logits = logits[
            :,
            -1,
            :
        ]


        # Temperature
        logits = (
            logits
            /
            max(
                temperature,
                1e-5
            )
        )


        # ----------------------------------------------------
        # Top-k
        # ----------------------------------------------------

        if top_k is not None:

            k = min(
                top_k,
                logits.size(-1)
            )

            values, _ = torch.topk(
                logits,
                k
            )

            minimum_value = (
                values[
                    :,
                    -1
                ].unsqueeze(-1)
            )

            logits = torch.where(
                logits
                <
                minimum_value,
                torch.full_like(
                    logits,
                    float("-inf")
                ),
                logits
            )


        # ----------------------------------------------------
        # Top-p
        # ----------------------------------------------------

        if top_p is not None:

            sorted_logits, sorted_indices = (
                torch.sort(
                    logits,
                    descending=True
                )
            )

            sorted_probabilities = F.softmax(
                sorted_logits,
                dim=-1
            )

            cumulative_probabilities = (
                torch.cumsum(
                    sorted_probabilities,
                    dim=-1
                )
            )

            sorted_indices_to_remove = (
                cumulative_probabilities
                >
                top_p
            )

            sorted_indices_to_remove[
                :,
                0
            ] = False

            indices_to_remove = (
                torch.zeros_like(
                    logits,
                    dtype=torch.bool
                )
            )

            indices_to_remove.scatter_(
                1,
                sorted_indices,
                sorted_indices_to_remove
            )

            logits = logits.masked_fill(
                indices_to_remove,
                float("-inf")
            )


        # ----------------------------------------------------
        # Sample next token
        # ----------------------------------------------------

        probabilities = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )


        input_ids = torch.cat(
            [
                input_ids,
                next_token
            ],
            dim=1
        )


        # EOS
        if (
            next_token.item()
            ==
            EOS_ID
        ):

            break


    # --------------------------------------------------------
    # Decode
    # --------------------------------------------------------

    decoded = tokenizer.decode(
        input_ids[
            0
        ].tolist(),
        skip_special_tokens=True
    )

    # Remove tokenizer's whitespace marker
    decoded = decoded.replace(
        "Ġ",
        ""
    )

    return decoded


# ============================================================
# 39. GENERATION TEST
# ============================================================

prompts = [

    "What are the fundamental rights guaranteed by the Constitution of India?",

    "What is the role of the Supreme Court of India in interpreting the Constitution?",

    "What are the grounds on which a person can file a writ petition?",

    "What is meant by natural justice in Indian law?",

    "What is the difference between civil and criminal liability?",

    "When can a court grant bail to an accused person?",

    "What are the essential elements required to establish negligence?",

    "What powers does a High Court have under Article 226 of the Constitution?",

    "What is the meaning of Article 21 of the Constitution of India?",

    "What is the principle of res judicata under Indian law?"

]


for prompt in prompts:

    print("\n")
    print("=" * 70)

    print(
        "PROMPT:"
    )

    print(
        prompt
    )

    print("-" * 70)

    generated = generate(
        prompt,
        max_new_tokens=150,
        temperature=0.8,
        top_k=50,
        top_p=0.95
    )

    print(
        generated
    )


# ============================================================
# 40. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("MAMBA TRAINING SUMMARY")
print("=" * 70)

print(
    f"Parameters:              "
    f"{total_params:,}"
)

print(
    f"Training sequences:      "
    f"{len(train_dataset):,}"
)

print(
    f"Validation sequences:    "
    f"{len(val_dataset):,}"
)

print(
    f"Context length:          "
    f"{CONTEXT_LENGTH}"
)

print(
    f"Vocabulary size:         "
    f"{VOCAB_SIZE:,}"
)

print(
    f"Layers:                  "
    f"{NUM_LAYERS}"
)

print(
    f"d_model:                 "
    f"{D_MODEL}"
)

print(
    f"d_state:                 "
    f"{D_STATE}"
)

print(
    f"d_conv:                  "
    f"{D_CONV}"
)

print(
    f"Epochs:                  "
    f"{EPOCHS}"
)

print(
    f"Effective batch size:    "
    f"{BATCH_SIZE * GRAD_ACCUMULATION_STEPS}"
)

print(
    f"Learning rate:           "
    f"{LEARNING_RATE}"
)

print(
    f"Warmup ratio:            "
    f"{WARMUP_RATIO}"
)

print(
    f"Best epoch:              "
    f"{best_epoch + 1}"
)

print(
    f"Best validation loss:    "
    f"{final_val_loss:.4f}"
)

print(
    f"Best validation PPL:     "
    f"{final_val_ppl:.2f}"
)

print(
    "\nModel directory:"
)

print(
    MODEL_DIR
)

print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/40.1 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.3 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/559.1 kB ? eta -:--:--

   ━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.6/559.1 kB 2.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━ 235.5/559.1 kB 2.8 MB/s eta 0:00:01

...

Epoch 1/3 | Opt Step 6,700/29,076 | Batch 107,200/155,060 | Loss: 4.1264 | Recent Loss: 4.3650 | LR: 8.983e-05 | Grad Norm: 4.636 | Time: 689.3 min


Epoch 1/3 | Opt Step 6,800/29,076 | Batch 108,800/155,060 | Loss: 4.9880 | Recent Loss: 4.3934 | LR: 8.949e-05 | Grad Norm: 3.992 | Time: 699.4 min


Epoch 1/3 | Opt Step 6,900/29,076 | Batch 110,400/155,060 | Loss: 4.8991 | Recent Loss: 4.3688 | LR: 8.915e-05 | Grad Norm: 3.290 | Time: 709.4 min


Epoch 1/3 | Opt Step 7,000/29,076 | Batch 112,000/155,060 | Loss: 4.2386 | Recent Loss: 4.3661 | LR: 8.880e-05 | Grad Norm: 3.217 | Time: 719.5 min


KeyboardInterrupt: 